In [1]:
# importing libraries 
import numpy as np 
import pandas as pd 
import warnings 
warnings.filterwarnings(action="ignore")

In [2]:
# importing data files  
x_df=pd.read_excel(r"C:\Users\Lenovo\Documents\Interview_projects_only\Health_care_chatbot\Data\Sample_arvyax_reflective_dataset(training_data).xlsx")
y_df=pd.read_excel(r"C:\Users\Lenovo\Documents\Interview_projects_only\Health_care_chatbot\Data\arvyax_test_inputs_120.xlsx")

In [3]:
# concatenating both files 
df= pd.concat([x_df , y_df] , ignore_index=True )

In [4]:
df.describe

<bound method NDFrame.describe of          id                                       journal_text ambience_type  \
0         1  The ocean ambience helped me stop drifting and...         ocean   
1         2  I tried to relax during the forest ambience, y...        forest   
2         3  The forest session slowed my thoughts and I fe...        forest   
3         4  the mountain ambience was pleasant, though i c...      mountain   
4         5  The rain session gave me a pause, but the pres...          rain   
...     ...                                                ...           ...   
1315  10116  ended up unable to come down from the day. the...          rain   
1316  10117  somehow i felt pretty grounded, but it took ti...        forest   
1317  10118  started off ready to start work, but i stayed ...          rain   
1318  10119  somehow i felt somewhat settled but still unea...        forest   
1319  10120  by the end i was distracted most of the time. ...      mountain   

     

In [5]:
df[:2]

,id,journal_text,ambience_type,duration_min,sleep_hours,energy_level,stress_level,time_of_day,previous_day_mood,face_emotion_hint,reflection_quality,emotional_state,intensity
0,1,The ocean ambience helped me stop drifting and...,ocean,12,6.5,4,2,afternoon,mixed,calm_face,clear,focused,3.0
1,2,"I tried to relax during the forest ambience, y...",forest,35,6.0,2,4,evening,calm,tired_face,vague,restless,3.0


In [6]:
df.intensity.value_counts()

intensity
4.0    277
3.0    240
5.0    229
2.0    228
1.0    226
Name: count, dtype: int64

----

Handling all null values in our data 

In [7]:
df.isnull().sum()

id                      0
journal_text            0
ambience_type           0
duration_min            0
sleep_hours             7
energy_level            0
stress_level            0
time_of_day             0
previous_day_mood      25
face_emotion_hint     142
reflection_quality      0
emotional_state       120
intensity             120
dtype: int64

In [8]:
# Filling empty rows with mean of the column
df['sleep_hours'].fillna(df['sleep_hours'].mean() , inplace =True)

In [9]:
# Dropping the null rows 
df = df.dropna(subset=['previous_day_mood'])

In [10]:
# Filling the empty values with 'none' , safe side for managing categorical data type 
df['face_emotion_hint'].fillna("none", inplace=True)

In [11]:
# Filling the empty values with 'mixed' , safe side for managing categorical data type 
df['emotional_state'].fillna("mixed", inplace=True)

In [12]:
# Filling empty rows with mean of the column 
df['intensity'].fillna(df['intensity'].mean() , inplace =True)

In [13]:
# rechecking null values 
df.isnull().sum()

id                    0
journal_text          0
ambience_type         0
duration_min          0
sleep_hours           0
energy_level          0
stress_level          0
time_of_day           0
previous_day_mood     0
face_emotion_hint     0
reflection_quality    0
emotional_state       0
intensity             0
dtype: int64

In [14]:
df.drop('id', axis = 1 , inplace =True)

In [15]:
df[:2]

,journal_text,ambience_type,duration_min,sleep_hours,energy_level,stress_level,time_of_day,previous_day_mood,face_emotion_hint,reflection_quality,emotional_state,intensity
0,The ocean ambience helped me stop drifting and...,ocean,12,6.5,4,2,afternoon,mixed,calm_face,clear,focused,3.0
1,"I tried to relax during the forest ambience, y...",forest,35,6.0,2,4,evening,calm,tired_face,vague,restless,3.0


In [16]:
# Converting the categorical data into True

In [17]:
# df = pd.get_dummies(df, columns=['ambience_type' , 'time_of_day' , 'previous_day_mood' , 'face_emotion_hint' , 'reflection_quality' ], drop_first=True)

In [18]:
# Enhancing Stress level into 3 classes to improve the accurary of stress_model
def simplify_stress(x):
    if x <= 2:
        return "low"
    elif x == 3:
        return "medium"
    else:
        return "high"

df["stress_level"] = df["stress_level"].apply(simplify_stress)

In [19]:
# Lable Encoding Target variable 
from sklearn.preprocessing import LabelEncoder
le_emotional_state = LabelEncoder()
le_ambience_type = LabelEncoder()
le_stress_level = LabelEncoder()
le_time_of_day = LabelEncoder()
le_previous_day_mood = LabelEncoder()
le_face_emotion_hint = LabelEncoder()
le_reflection_quality = LabelEncoder()

In [20]:
df['emotional_state']=le_emotional_state.fit_transform(df['emotional_state'] )
df['ambience_type']=le_ambience_type.fit_transform(df['ambience_type'])
df['stress_level']=le_stress_level.fit_transform(df['stress_level'])
df['time_of_day']=le_time_of_day.fit_transform(df['time_of_day'])
df['previous_day_mood']=le_previous_day_mood.fit_transform(df['previous_day_mood'])
df['face_emotion_hint']=le_face_emotion_hint.fit_transform(df['face_emotion_hint'])
df['reflection_quality']=le_reflection_quality.fit_transform(df['reflection_quality'])

In [21]:
df[:2]

,journal_text,ambience_type,duration_min,sleep_hours,energy_level,stress_level,time_of_day,previous_day_mood,face_emotion_hint,reflection_quality,emotional_state,intensity
0,The ocean ambience helped me stop drifting and...,3,12,6.5,4,1,0,2,0,0,1,3.0
1,"I tried to relax during the forest ambience, y...",1,35,6.0,2,0,2,0,5,2,5,3.0


In [22]:
df_st = df

-----

In [23]:
# Importing text vectorizer for "journal_text" processing 
from sklearn.feature_extraction.text import TfidfVectorizer

In [24]:
tfidf = TfidfVectorizer(max_features=500 , stop_words='english' )

In [25]:
text_features = tfidf.fit_transform(df['journal_text'])

In [26]:
text_df = pd.DataFrame(text_features.toarray() , columns = tfidf.get_feature_names_out())

In [27]:
text_df

,able,action,actually,affected,ambience,anotehr,anxious,attention,audio,aware,...,wants,wasn,weirdly,window,woke,work,worked,worse,wound,yesterday
0,0.0,0.0,0.0,0.0,0.249779,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.252706,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.299996,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1290,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
1291,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
1292,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.413755,0.0,0.0,0.0,0.0
1293,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0


In [28]:
df = df.drop(columns=['journal_text'])

In [29]:
df = pd.concat([df.reset_index(drop=True), text_df.reset_index(drop=True)], axis=1)

In [30]:
df

,ambience_type,duration_min,sleep_hours,energy_level,stress_level,time_of_day,previous_day_mood,face_emotion_hint,reflection_quality,emotional_state,...,wants,wasn,weirdly,window,woke,work,worked,worse,wound,yesterday
0,3,12,6.50000,4,1,0,2,0,0,1,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
1,1,35,6.00000,2,0,2,0,5,2,5,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
2,1,3,6.00495,2,1,4,4,1,0,0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
3,2,25,7.00000,4,0,4,1,0,2,3,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
4,1,12,8.00000,3,1,3,2,0,2,0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1290,4,7,5.00000,4,1,4,1,3,1,2,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
1291,1,5,7.00000,1,2,0,4,4,0,2,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
1292,4,12,8.00000,4,1,0,0,5,2,2,...,0.0,0.0,0.0,0.0,0.0,0.413755,0.0,0.0,0.0,0.0
1293,1,25,8.50000,4,0,4,3,5,1,2,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0


Now , our data is ready .

----

Splitting data for different target variables and training models 

In [31]:
from xgboost import XGBClassifier

In [32]:
em_model = XGBClassifier(n_estimators=200,max_depth=5,learning_rate=0.1,)

Modeling for Emotional state prediction 

In [33]:
# Dividing data into Input and Target columns 
x_em = df.drop(["emotional_state","stress_level"] , axis = 1)
y_em = df.emotional_state

In [34]:
# Importing train test split library 
from sklearn.model_selection import train_test_split

In [35]:
x_train , x_test , y_train , y_test = train_test_split(x_em , y_em , test_size=0.2 , random_state=42)

In [36]:
# Model training 
em_model.fit(x_train , y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=200,
              n_jobs=None, num_parallel_tree=None, ...)

In [37]:
y_pred = em_model.predict(x_test)

In [38]:
# Importing classification report 
from sklearn.metrics import classification_report

In [39]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.54      0.67      0.60        39
           1       0.63      0.68      0.66        28
           2       0.58      0.75      0.66        53
           3       0.69      0.50      0.58        44
           4       0.75      0.60      0.67        50
           5       0.68      0.60      0.64        45

    accuracy                           0.63       259
   macro avg       0.64      0.63      0.63       259
weighted avg       0.65      0.63      0.63       259



In [40]:
# Accuracy 63 % 

----

Modeling for State level prediction 

In [41]:
# Dividing data into Input and Target columns
x_st = x_em
y_st = df.stress_level

In [42]:
x_train , x_test , y_train , y_test = train_test_split(x_st , y_st , test_size=0.2 , random_state=42)

In [43]:
st_model = XGBClassifier(n_estimators=200,max_depth=5,learning_rate=0.1)

In [44]:
# Model training 
st_model.fit(x_train , y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=200,
              n_jobs=None, num_parallel_tree=None, ...)

In [45]:
y_pred = st_model.predict(x_test)

In [46]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.50      0.53      0.51       118
           1       0.38      0.48      0.42        92
           2       0.16      0.06      0.09        49

    accuracy                           0.42       259
   macro avg       0.35      0.35      0.34       259
weighted avg       0.39      0.42      0.40       259



In [47]:
# Accuracy 46  %

----

Dumping the models using Joblib

In [48]:
# Importing library
import joblib

In [49]:
joblib.dump(em_model , "em_model.pkl")

['em_model.pkl']

In [50]:
joblib.dump(st_model , "st_model.pkl")

['st_model.pkl']

In [51]:
joblib.dump(tfidf , "vectorizer.pkl")

['vectorizer.pkl']

In [52]:
joblib.dump(le_emotional_state , "le_emotional_state.pkl")

['le_emotional_state.pkl']

In [53]:
joblib.dump(le_ambience_type , "le_ambience_type.pkl")

['le_ambience_type.pkl']

In [54]:
joblib.dump(le_stress_level , "le_stress_level.pkl")

['le_stress_level.pkl']

In [55]:
joblib.dump(le_time_of_day , "le_time_of_day.pkl")

['le_time_of_day.pkl']

In [56]:
joblib.dump(le_previous_day_mood , "le_previous_day_mood.pkl")

['le_previous_day_mood.pkl']

In [57]:
joblib.dump(le_face_emotion_hint , "le_face_emotion_hint.pkl")

['le_face_emotion_hint.pkl']

In [58]:
joblib.dump(le_reflection_quality , "le_reflection_quality.pkl")

['le_reflection_quality.pkl']

---- 